# 二阶段分类数据集离线增强

这个 notebook 用于在不覆盖原始数据集的前提下，生成一份增强版分类数据集。

默认策略：
- 保留 `val/test` 不变
- 只对 `train/abnormal` 做定向增强
- 输出到新的增强版目录


## 0. 依赖导入


In [ ]:
from pathlib import Path
import json
import random
import shutil
from collections import Counter

from PIL import Image, ImageEnhance, ImageFilter

print('✓ 依赖导入完成')


## 1. 核心函数


In [ ]:
def count_split_classes(dataset_dir):
    dataset_dir = Path(dataset_dir)
    split_counts = {}
    for split_dir in sorted([path for path in dataset_dir.iterdir() if path.is_dir()]):
        class_counts = {}
        for class_dir in sorted([path for path in split_dir.iterdir() if path.is_dir()]):
            class_counts[class_dir.name] = len([p for p in class_dir.iterdir() if p.is_file()])
        split_counts[split_dir.name] = class_counts
    return split_counts


def apply_single_augmentation(image, variant_index, rng):
    augmented = image.copy()
    recipe = []

    if variant_index % 2 == 0:
        augmented = augmented.transpose(Image.FLIP_LEFT_RIGHT)
        recipe.append('fliplr')

    rotation = rng.choice([-8, -5, -3, 3, 5, 8])
    augmented = augmented.rotate(rotation, resample=Image.Resampling.BICUBIC, expand=False, fillcolor='white')
    recipe.append(f'rot{rotation}')

    brightness = rng.uniform(0.88, 1.12)
    augmented = ImageEnhance.Brightness(augmented).enhance(brightness)
    recipe.append(f'bri{brightness:.2f}')

    contrast = rng.uniform(0.88, 1.15)
    augmented = ImageEnhance.Contrast(augmented).enhance(contrast)
    recipe.append(f'con{contrast:.2f}')

    if variant_index % 3 == 0:
        augmented = augmented.filter(ImageFilter.SHARPEN)
        recipe.append('sharp')
    elif variant_index % 3 == 1:
        augmented = augmented.filter(ImageFilter.GaussianBlur(radius=0.5))
        recipe.append('blur')

    return augmented, '-'.join(recipe)


def build_augmented_dataset(
    source_dataset_dir,
    output_dataset_dir,
    target_class='abnormal',
    copies_per_image=2,
    augment_abnormal_only=True,
    random_seed=42,
):
    source_dataset_dir = Path(source_dataset_dir)
    output_dataset_dir = Path(output_dataset_dir)

    if output_dataset_dir.exists():
        shutil.rmtree(output_dataset_dir)
    shutil.copytree(source_dataset_dir, output_dataset_dir)

    rng = random.Random(random_seed)
    augmented_counts = Counter()

    train_dir = output_dataset_dir / 'train'
    for class_dir in sorted([path for path in train_dir.iterdir() if path.is_dir()]):
        if augment_abnormal_only and class_dir.name != target_class:
            continue

        source_images = sorted([path for path in class_dir.iterdir() if path.is_file()])
        for image_path in source_images:
            with Image.open(image_path) as image:
                rgb_image = image.convert('RGB')
                for variant_index in range(copies_per_image):
                    augmented, recipe = apply_single_augmentation(rgb_image, variant_index, rng)
                    output_name = f"{image_path.stem}_aug{variant_index + 1:02d}_{recipe}.jpg"
                    augmented.save(class_dir / output_name, quality=95)
                    augmented_counts[class_dir.name] += 1

    summary = {
        'source_dataset_dir': str(source_dataset_dir),
        'output_dataset_dir': str(output_dataset_dir),
        'augment_abnormal_only': augment_abnormal_only,
        'target_class': target_class,
        'copies_per_image': copies_per_image,
        'augmented_counts': dict(augmented_counts),
        'final_split_counts': count_split_classes(output_dataset_dir),
    }

    (output_dataset_dir / 'augmentation_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return summary


print('✓ 核心函数准备完成')


## 2. 路径与参数配置


In [ ]:
SOURCE_DATASET_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/insulator_cls_dataset_tight')
OUTPUT_DATASET_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/insulator_cls_dataset_augmented')

AUGMENT_ABNORMAL_ONLY = True
TARGET_CLASS = 'abnormal'
COPIES_PER_IMAGE = 2
RANDOM_SEED = 42

print('SOURCE_DATASET_DIR =', SOURCE_DATASET_DIR)
print('OUTPUT_DATASET_DIR =', OUTPUT_DATASET_DIR)


## 3. 查看增强前分布


In [ ]:
before_counts = count_split_classes(SOURCE_DATASET_DIR)
print(json.dumps(before_counts, ensure_ascii=False, indent=2))


## 4. 生成增强版数据集


In [ ]:
augmentation_summary = build_augmented_dataset(
    source_dataset_dir=SOURCE_DATASET_DIR,
    output_dataset_dir=OUTPUT_DATASET_DIR,
    target_class=TARGET_CLASS,
    copies_per_image=COPIES_PER_IMAGE,
    augment_abnormal_only=AUGMENT_ABNORMAL_ONLY,
    random_seed=RANDOM_SEED,
)

print(json.dumps(augmentation_summary, ensure_ascii=False, indent=2))


## 5. 使用提示


In [ ]:
print('增强版数据集目录:')
print(OUTPUT_DATASET_DIR)
print('\n接下来在 train_insulator_classifier.ipynb 中把 DATASET_DIR 改成这个目录即可。')
